# Pareto-DPO: Multi-Objective Preference Alignment for Scaffold Decoration

Step-by-step walkthrough of the full pipeline.

In [ ]:
import sys
sys.path.append('..')

from pareto_dpo.config import ParetoDPOConfig
from pareto_dpo.data.dataset import read_smiles_file, ScaffoldDataset
from pareto_dpo.model.gpt import ScaffoldGPT
from pareto_dpo.data.tokenizer import load_or_create_tokenizer
from pareto_dpo.optimization.scorer import compute_objectives
from pareto_dpo.optimization.pareto import build_pareto_preference_pairs_batched
from pareto_dpo.optimization.dpo_trainer import DPOTrainer
from pareto_dpo.evaluation.metrics import evaluate_generation

import matplotlib.pyplot as plt
import numpy as np
from rdkit import Chem
from rdkit.Chem import Draw

In [ ]:
config = ParetoDPOConfig()
smiles = read_smiles_file('../data/chembl_demo.smi', max_mols=50000)
print(f'Loaded {len(smiles)} molecules')

In [ ]:
tokenizer = load_or_create_tokenizer(smiles, 'data/tokenizer.json')
model = ScaffoldGPT.from_pretrained('gpt2', tokenizer)
print(f'Vocab size: {len(tokenizer)}')

In [ ]:
generated = model.generate_from_scaffold('c1ccccc1', num_return_sequences=5)
for i, smi in enumerate(generated):
    scores = compute_objectives(smi, config.objectives)
    print(f'{i}: {smi}  QED={scores["qed"]:.3f}  cLogP={scores["clogp"]:.2f}  SA={scores["sa"]:.2f}  MW={scores["mw"]:.0f}')

In [ ]:
pairs = build_pareto_preference_pairs_batched(
    scaffolds=['c1ccccc1', 'c1ccncc1', 'c1ccc2ccccc2c1'],
    generate_fn=lambda s, n: model.generate_from_scaffold(s, num_return_sequences=n),
    objectives=config.objectives,
    directions=config.objective_directions,
    num_samples_per_scaffold=64,
    verbose=True,
)
print(f'Built {len(pairs)} Pareto preference pairs')

In [ ]:
ref_model = ScaffoldGPT.from_pretrained('gpt2', tokenizer)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad = False

trainer = DPOTrainer(model, ref_model, tokenizer, config)
trainer.train(pairs)

In [ ]:
scaffolds_test = ['c1ccccc1', 'c1ccncc1', 'c1ccc2ccccc2c1']
metrics_before = evaluate_generation(ref_model, scaffolds_test, smiles, 64, config.objectives, config.objective_directions)
metrics_after = evaluate_generation(model, scaffolds_test, smiles, 64, config.objectives, config.objective_directions)

print('Before DPO:')
for k, v in metrics_before.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')
print()
print('After DPO:')
for k, v in metrics_after.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
objectives = config.objectives
for idx, obj in enumerate(objectives):
    ax = axes[idx // 2, idx % 2]
    ax.bar(['Before DPO', 'After DPO'], [metrics_before[f'mean_{obj}'], metrics_after[f'mean_{obj}']])
    ax.set_title(f'Mean {obj.upper()}')
plt.tight_layout()
plt.show()